In [ ]:
from __future__ import annotations
import hashlib
import json
import math
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import FashionMNIST, MNIST
from IPython.display import Image, display


In [ ]:
cfg = json.loads(r'''
{
  "dataset": "FashionMNIST",
  "seed": 42,
  "batch_size": 128,
  "num_workers": 0,
  "device": "auto",
  "precision": "float32",
  "data_dir": "data",
  "output_dir": "artifacts/fashion",
  "latent_dim": 64,
  "noise_dim": 100,
  "sparse_lambda": 0.0001,
  "sparsity_threshold": 0.001,
  "ddpm_steps": 400,
  "ddpm_channels": 32,
  "beta_start": 0.00025,
  "beta_end": 0.05,
  "lr": 0.001,
  "gan_lr": 0.0002,
  "ddpm_lr": 0.0002,
  "epochs": {"mlp": 25, "ae": 20, "sparse_ae": 20, "vae": 25, "gan": 40, "dcgan": 40, "ddpm": 30, "decoder": 20},
  "fid_samples": 7000,
  "fid_batch_size": 64,
  "timing_warmup": 3,
  "timing_repeats": 10,
  "normalization": {"ae": "[0,1]", "sparse_ae": "[0,1]", "vae": "[0,1]", "mlp": "[0,1]", "decoder": "[0,1]", "gan": "[-1,1]", "dcgan": "[-1,1]", "ddpm": "[-1,1]"},
  "checkpoint_selection": {"mlp": "maximum validation accuracy", "ae": "minimum validation objective", "sparse_ae": "minimum validation objective", "vae": "minimum validation ELBO loss", "decoder": "minimum validation MSE", "ddpm": "minimum fixed-noise validation MSE", "gan": "last epoch of predeclared budget", "dcgan": "last epoch of predeclared budget"}
}
''')

RUN_MLP = False


In [ ]:
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.set_num_threads(min(8, os.cpu_count() or 1))

def get_device(config):
    name = config.get("device", "auto")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu") if name == "auto" else torch.device(name)

seed_all(cfg["seed"])
device = get_device(cfg)
print("Device:", device, "| Dataset:", cfg["dataset"], "| Seed:", cfg["seed"])


In [ ]:
class DataModule:
    """Pool official 60k + 10k, then persist one random 80/10/10 split."""

    def __init__(self, config, download=True):
        self.config = config
        dataset_cls = {"FashionMNIST": FashionMNIST, "MNIST": MNIST}[config["dataset"]]
        root = Path(config["data_dir"])
        official_train = dataset_cls(root, train=True, download=download)
        official_test = dataset_cls(root, train=False, download=download)
        self.images = torch.cat([official_train.data, official_test.data]).unsqueeze(1)
        self.labels = torch.cat([official_train.targets, official_test.targets])
        assert self.images.shape == (70000, 1, 28, 28)
        self.split_path = root / f'{config["dataset"]}_split_seed{config["seed"]}.npz'
        if not self.split_path.exists():
            order = np.random.default_rng(config["seed"]).permutation(70000)
            np.savez(self.split_path, train=order[:56000], validation=order[56000:63000], test=order[63000:])
        with np.load(self.split_path) as stored:
            self.indices = {k: stored[k].copy() for k in ("train", "validation", "test")}
        expected = (56000, 7000, 7000)
        assert tuple(len(self.indices[k]) for k in self.indices) == expected
        all_indices = np.concatenate(list(self.indices.values()))
        assert np.array_equal(np.sort(all_indices), np.arange(70000)), "Split overlaps or is incomplete"
        self.split_sha256 = hashlib.sha256(self.split_path.read_bytes()).hexdigest()

    def loader(self, split, epoch=0, batch_size=None):
        idx = torch.from_numpy(self.indices[split])
        dataset = TensorDataset(self.images[idx], self.labels[idx], idx)
        generator = torch.Generator().manual_seed(self.config["seed"] + epoch)
        return DataLoader(dataset, batch_size=batch_size or self.config["batch_size"],
                          shuffle=split == "train", num_workers=self.config["num_workers"],
                          pin_memory=torch.cuda.is_available(), generator=generator, drop_last=False)

    def summary(self):
        return {"dataset": self.config["dataset"], "pool_order": "torchvision train then test",
                "split_algorithm": "numpy default_rng(seed).permutation(70000)",
                "seed": self.config["seed"], "sizes": {k: len(v) for k, v in self.indices.items()},
                "split_file": str(self.split_path), "split_sha256": self.split_sha256,
                "pixel_storage": "uint8 [0,255]; converted per batch to float32 [0,1]",
                "disjoint_and_complete": True}

def to_device(batch, device):
    images, labels, indices = batch
    return images.to(device, non_blocking=True).float().div_(255), labels.to(device), indices


data = None
if RUN_MLP:
    data = DataModule(cfg)
    print(data.summary())


In [ ]:
class Classifier(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, latent)
        self.fc3 = nn.Linear(latent, 10)

    def encode(self, x):
        return F.relu(self.fc2(F.relu(self.fc1(x.flatten(1)))))

    def forward(self, x):
        return self.fc3(self.encode(x))

    def transpose_latent(self, z):
        # For row vectors, z @ W2 then @ W1 is the column-vector W1^T ReLU(W2^T z).
        return torch.sigmoid(F.relu(z @ self.fc2.weight) @ self.fc1.weight).reshape(-1, 1, 28, 28)

    def transpose_class(self, one_hot):
        return self.transpose_latent(F.relu(one_hot @ self.fc3.weight))

class Decoder(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(latent, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid())

    def forward(self, z):
        return self.net(z).reshape(-1, 1, 28, 28)


In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                                     nn.Linear(256, latent), nn.ReLU())
        self.decoder = Decoder(latent)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

class VAE(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU())
        self.mu = nn.Linear(256, latent)
        self.logvar = nn.Linear(256, latent)
        self.decoder = Decoder(latent)

    def encode(self, x):
        h = self.encoder(x)
        return self.mu(h), self.logvar(h)

    def forward(self, x, deterministic=False):
        mu, logvar = self.encode(x)
        z = mu if deterministic else mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.decoder(z), mu, logvar


In [ ]:
class VanillaGAN(nn.Module):
    def __init__(self, noise=100):
        super().__init__()
        self.generator = nn.Sequential(nn.Linear(noise, 256), nn.LeakyReLU(0.2),
                                       nn.Linear(256, 512), nn.LeakyReLU(0.2),
                                       nn.Linear(512, 784), nn.Tanh(), nn.Unflatten(1, (1, 28, 28)))
        self.discriminator = nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.LeakyReLU(0.2),
                                           nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Linear(256, 1))

    def forward(self, z):
        return self.generator(z)

class DCGAN(nn.Module):
    def __init__(self, noise=100):
        super().__init__()
        self.generator = nn.Sequential(
            nn.Linear(noise, 128 * 7 * 7), nn.Unflatten(1, (128, 7, 7)),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 1, 4, 2, 1), nn.Tanh())
        self.discriminator = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Flatten(), nn.Linear(128 * 7 * 7, 1))
        self.apply(self._init)

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
            nn.init.normal_(module.weight, 0, 0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.BatchNorm2d):
            nn.init.normal_(module.weight, 1, 0.02)
            nn.init.zeros_(module.bias)

    def forward(self, z):
        return self.generator(z)


In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.net = nn.Sequential(nn.Linear(dim, dim * 2), nn.SiLU(), nn.Linear(dim * 2, dim))

    def forward(self, t):
        freq = torch.exp(torch.arange(self.dim // 2, device=t.device) * (-math.log(10000) / (self.dim // 2 - 1)))
        angles = t.float()[:, None] * freq[None]
        return self.net(torch.cat([angles.sin(), angles.cos()], dim=1))

class TimeBlock(nn.Module):
    def __init__(self, inp, out, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(inp, out, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out)
        self.time = nn.Linear(time_dim, out)
        self.conv2 = nn.Conv2d(out, out, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out)
        self.skip = nn.Conv2d(inp, out, 1) if inp != out else nn.Identity()

    def forward(self, x, emb):
        h = F.silu(self.norm1(self.conv1(x))) + self.time(emb)[:, :, None, None]
        h = F.silu(self.norm2(self.conv2(h)))
        return h + self.skip(x)

class Denoiser(nn.Module):
    """Small U-Net: 28 -> 14 -> 7 -> 14 -> 28; timestep injection in every block."""
    def __init__(self, channels=32):
        super().__init__()
        c = channels
        self.time = TimeEmbedding(c * 4)
        self.down1 = TimeBlock(1, c, c * 4)
        self.down2 = TimeBlock(c, c * 2, c * 4)
        self.mid = TimeBlock(c * 2, c * 4, c * 4)
        self.up2 = TimeBlock(c * 6, c * 2, c * 4)
        self.up1 = TimeBlock(c * 3, c, c * 4)
        self.out = nn.Conv2d(c, 1, 1)

    def forward(self, x, t):
        e = self.time(t)
        a = self.down1(x, e)
        b = self.down2(F.avg_pool2d(a, 2), e)
        h = self.mid(F.avg_pool2d(b, 2), e)
        h = self.up2(torch.cat([F.interpolate(h, size=b.shape[-2:], mode="nearest"), b], 1), e)
        h = self.up1(torch.cat([F.interpolate(h, size=a.shape[-2:], mode="nearest"), a], 1), e)
        return self.out(h)

class DDPM(nn.Module):
    def __init__(self, steps=200, channels=32, beta_start=1e-4, beta_end=0.02):
        super().__init__()
        self.denoiser = Denoiser(channels)
        self.steps = steps
        beta = torch.linspace(beta_start, beta_end, steps)
        alpha = 1 - beta
        abar = torch.cumprod(alpha, 0)
        prev = F.pad(abar[:-1], (1, 0), value=1)
        for name, value in {"beta": beta, "alpha": alpha, "abar": abar,
                            "posterior_variance": beta * (1 - prev) / (1 - abar)}.items():
            self.register_buffer(name, value)

    def q_sample(self, x, t, noise):
        a = self.abar[t, None, None, None]
        return a.sqrt() * x + (1 - a).sqrt() * noise

    def forward(self, x, t):
        return self.denoiser(x, t)

    @torch.no_grad()
    def sample(self, x, generator=None, snapshots=False):
        chain = {self.steps: x[:1].detach().cpu()} if snapshots else {}
        keep = {0, self.steps // 16, self.steps // 4, self.steps // 2, 3 * self.steps // 4}
        for i in reversed(range(self.steps)):
            t = torch.full((len(x),), i, device=x.device, dtype=torch.long)
            eps = self.denoiser(x, t)
            # Predict/clamp x0 and use the exact DDPM posterior mean.
            x0 = ((x - (1 - self.abar[i]).sqrt() * eps) / self.abar[i].sqrt()).clamp(-1, 1)
            prev = self.abar[i - 1] if i else torch.ones_like(self.abar[i])
            mean = (prev.sqrt() * self.beta[i] / (1 - self.abar[i])) * x0
            mean += ((1 - prev) * self.alpha[i].sqrt() / (1 - self.abar[i])) * x
            noise = torch.randn(x.shape, device=x.device, generator=generator) if i else torch.zeros_like(x)
            x = mean + self.posterior_variance[i].sqrt() * noise
            if snapshots and i in keep:
                chain[i] = x[:1].detach().cpu()
        return (x, chain) if snapshots else x

def build_model(name, config):
    latent, noise = config["latent_dim"], config["noise_dim"]
    if name in ("ae", "sparse_ae"):
        return Autoencoder(latent)
    if name == "vae":
        return VAE(latent)
    if name == "mlp":
        return Classifier(latent)
    if name == "decoder":
        return Decoder(latent)
    if name == "gan":
        return VanillaGAN(noise)
    if name == "dcgan":
        return DCGAN(noise)
    if name == "ddpm":
        return DDPM(config["ddpm_steps"], config["ddpm_channels"], config["beta_start"], config["beta_end"])
    raise ValueError(name)


In [ ]:
def load_trained(name):
    path = Path(cfg["output_dir"]) / name / "best.pt"
    saved = torch.load(path, map_location="cpu", weights_only=False)
    if saved["config"] != cfg or saved["split_sha256"] != data.split_sha256:
        raise ValueError(f"Config/split checkpoint {name} tidak cocok")
    if saved.get("max_batches") is not None:
        raise ValueError("Checkpoint smoke tidak boleh menjadi hasil final")
    if name == "decoder":
        classifier_path = Path(cfg["output_dir"]) / "mlp/best.pt"
        if saved["classifier_sha256"] != hashlib.sha256(classifier_path.read_bytes()).hexdigest():
            raise ValueError("Decoder tidak cocok dengan classifier")
    model = build_model(name, cfg).to(device)
    model.load_state_dict(saved["model"])
    return model.eval().requires_grad_(False)


In [ ]:
mlp_demo = None
if RUN_MLP:
    classifier = load_trained("mlp")
    decoder = load_trained("decoder")
    ae = load_trained("ae")
    features, correct, total = [], 0, 0
    with torch.inference_mode():
        for batch in data.loader("test"):
            x, y, _ = to_device(batch, device)
            z = classifier.encode(x)
            features.append(z.cpu())
            correct += (classifier.fc3(z).argmax(1) == y).sum().item()
            total += len(x)
        features = torch.cat(features)
        x, _, _ = to_device(next(iter(data.loader("test"))), device)
        x = x[:10]
        mu = features.mean(0).to(device)
        sigma = features.std(0, correction=0).to(device)
        generator = torch.Generator(device=device).manual_seed(cfg["seed"] + 14000)
        z_random = mu + sigma * torch.randn(16, cfg["latent_dim"], device=device, generator=generator)
        mlp_demo = {
            "accuracy": correct / total, "n_test": total,
            "W1_shape": list(classifier.fc1.weight.shape),
            "W2_shape": list(classifier.fc2.weight.shape),
            "W3_shape": list(classifier.fc3.weight.shape),
            "original": x.cpu(),
            "transpose_class": classifier.transpose_class(torch.eye(10, device=device)).cpu(),
            "transpose_latent": classifier.transpose_latent(features[:10].to(device)).cpu(),
            "frozen_decoder": decoder(features[:10].to(device)).cpu(),
            "vanilla_ae": ae(x)[0].cpu(),
            "gaussian_samples": decoder(z_random).cpu()
        }
    print(f"Akurasi test: {100 * mlp_demo['accuracy']:.2f}% dari {total} citra")
    print("W1/W2/W3:", mlp_demo["W1_shape"], mlp_demo["W2_shape"], mlp_demo["W3_shape"])
else:
    print("MLP dilewati. Aktifkan RUN_MLP setelah checkpoint tepercaya dan split asli tersedia.")


In [ ]:
metrics_file = Path("results/metrics.csv")
if metrics_file.exists():
    metrics = pd.read_csv(metrics_file, keep_default_na=False)
    display(metrics)
else:
    print("Tabel metrik belum tersedia.")


In [ ]:
figure_names = [
    "figure_1_reconstruction", "figure_2_sampling", "figure_3_mlp",
    "figure_4_interpolation", "figure_5_curves_trajectory"
]
for name in figure_names:
    path = Path("results") / f"{name}.png"
    if path.exists():
        print(name)
        display(Image(filename=str(path)))
    else:
        print("Figur belum tersedia:", name)
